# Task 2: Medical Fine-Tuning with QLoRA using Unsloth in Google Colab

> **Assignment:** Generative AI Internship – Task 2  
> **Goal:** Fine-tune a base LLM (Llama-3 or DeepSeek-R1) on a medical Q&A dataset using QLoRA + Unsloth

### What you'll learn
- 4-bit quantised (QLoRA) low-rank adapter fine-tuning
- PEFT / LoRA adapter setup and configuration
- Tokenisation, dataset formatting, epoch-based training
- Monitoring GPU memory in Colab
- Saving, merging & testing your fine-tuned adapter

**Runtime:** GPU (T4 or better) — `Runtime → Change runtime type → T4 GPU`

## Step 1 — Install Dependencies

In [ ]:
# Install Unsloth and its dependencies (optimised for Colab)
# This cell takes ~2-3 minutes on first run
!pip install unsloth --quiet
!pip install datasets transformers accelerate bitsandbytes peft trl --quiet
print("✅ All dependencies installed.")

## Step 2 — Verify GPU & Memory

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ GPU not available — change runtime to GPU!"

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU detected: {gpu}")
print(f"   VRAM available: {vram:.1f} GB")

## Step 3 — Load Base Model with Unsloth (4-bit Quantisation)

In [ ]:
from unsloth import FastLanguageModel

# ── Configuration ──────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048        # Maximum context length
DTYPE          = None        # Auto-detect (bfloat16 on Ampere+, float16 elsewhere)
LOAD_IN_4BIT   = True        # Enable 4-bit quantisation (saves ~75% VRAM)

# Supported base models — pick one:
#   "unsloth/Meta-Llama-3.1-8B"          (recommended, best quality)
#   "unsloth/DeepSeek-R1-Distill-Qwen-7B" (DeepSeek option)
#   "unsloth/Meta-Llama-3.1-8B-Instruct" (instruction-tuned base)
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B"

print(f"Loading {BASE_MODEL} with 4-bit quantisation...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

print(f"✅ Model loaded successfully!")
print(f"   Parameters: {model.num_parameters()/1e9:.2f}B")

## Step 4 — Attach LoRA Adapter (PEFT)

In [ ]:
# ── LoRA Hyperparameters ───────────────────────────────────────────────────
LORA_R         = 16     # Rank: higher = more capacity, more VRAM (try 8, 16, 32)
LORA_ALPHA     = 16     # Scaling factor (usually == r)
LORA_DROPOUT   = 0.05   # Dropout for regularisation

# Which linear layers to target (Unsloth recommends all four for best results)
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    model,
    r                       = LORA_R,
    target_modules          = TARGET_MODULES,
    lora_alpha              = LORA_ALPHA,
    lora_dropout            = LORA_DROPOUT,
    bias                    = "none",          # No bias for memory saving
    use_gradient_checkpointing = "unsloth",    # Unsloth's custom gradient checkpointing
    random_state            = 42,
    use_rslora              = False,           # Rank-stabilised LoRA (optional)
    loftq_config            = None,
)

# Report trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapter attached.")
print(f"   Trainable params : {trainable:,} ({100*trainable/total:.2f}% of total)")
print(f"   Total params     : {total:,}")

## Step 5 — Load & Prepare Medical Q&A Dataset

In [ ]:
from datasets import load_dataset

# ── Load a medical Q&A dataset from HuggingFace Hub ───────────────────────
# Options:
#  "medalpaca/medical_meadow_medical_flashcards"  — 33k clinical flashcards
#  "lavita/ChatDoctor-HealthCareMagic-100k"       — 100k doctor chat logs
#  "Shengding/Medical-QA"                         — general medical QA

DATASET_NAME = "medalpaca/medical_meadow_medical_flashcards"
print(f"Loading dataset: {DATASET_NAME}")

raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"✅ Dataset loaded: {len(raw_dataset):,} samples")
print("\nSample entry:")
print(raw_dataset[0])

In [ ]:
# ── Prompt template (Alpaca-style instruction format) ─────────────────────
ALPACA_TEMPLATE = """Below is a medical question. Provide a clear, accurate, and concise answer.

### Question:
{question}

### Answer:
{answer}{eos}"""

EOS_TOKEN = tokenizer.eos_token

def format_medical_prompt(examples):
    """
    Convert raw dataset rows to the Alpaca prompt format.
    Handles both 'input/output' and 'question/answer' column naming.
    """
    texts = []
    for i in range(len(examples.get("input", examples.get("question", [])))):
        question = examples.get("input",    examples.get("question", [""]))[i]
        answer   = examples.get("output",   examples.get("answer",   [""]))[i]
        texts.append(
            ALPACA_TEMPLATE.format(
                question=question,
                answer=answer,
                eos=EOS_TOKEN
            )
        )
    return {"text": texts}

# Apply formatting and tokenise
formatted_dataset = raw_dataset.map(
    format_medical_prompt,
    batched=True,
    remove_columns=raw_dataset.column_names
)

# Use a subset for faster training (remove [:5000] to use full dataset)
train_dataset = formatted_dataset.select(range(min(5000, len(formatted_dataset))))

print(f"✅ Dataset formatted: {len(train_dataset):,} training samples")
print("\nFormatted sample preview:")
print(train_dataset[0]["text"][:500], "...")

## Step 6 — Configure Training (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Training Hyperparameters ───────────────────────────────────────────────
EPOCHS          = 3             # Number of training epochs
BATCH_SIZE      = 2             # Per-device batch size (increase if VRAM allows)
GRAD_ACCUM      = 4             # Gradient accumulation steps (effective batch = 8)
LEARNING_RATE   = 2e-4          # LoRA learning rate
WARMUP_STEPS    = 10            # LR warmup steps
SAVE_STEPS      = 100           # Save checkpoint every N steps
OUTPUT_DIR      = "./medical_llm_output"

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset= train_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,     # Pack short sequences together (optional speed-up)
    args = TrainingArguments(
        per_device_train_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps                 = WARMUP_STEPS,
        num_train_epochs             = EPOCHS,
        learning_rate                = LEARNING_RATE,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        optim                        = "adamw_8bit",   # 8-bit Adam saves VRAM
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 42,
        output_dir                   = OUTPUT_DIR,
        save_steps                   = SAVE_STEPS,
        report_to                    = "none",         # Disable wandb for simplicity
    ),
)

print("✅ Trainer configured.")
print(f"   Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} effective")
print(f"   LR: {LEARNING_RATE}  |  Output: {OUTPUT_DIR}")

## Step 7 — Monitor VRAM Before Training

In [ ]:
import torch

def print_gpu_stats(label=""):
    """Print current GPU memory usage."""
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{'─'*45}")
    print(f"  GPU Memory [{label}]")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserved:.2f} GB")
    print(f"{'─'*45}")

print_gpu_stats("Before Training")

## Step 8 — Train the Model

In [ ]:
import time

print("🚀 Starting training...")
start = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes.")
print(f"   Final loss        : {trainer_stats.training_loss:.4f}")
print(f"   Total steps       : {trainer_stats.global_step}")
print(f"   Samples/second    : {trainer_stats.metrics.get('train_samples_per_second', 'N/A')}")

print_gpu_stats("After Training")

## Step 9 — Save the Fine-Tuned LoRA Adapter

In [ ]:
ADAPTER_DIR = "./medical_lora_adapter"

# Save adapter weights and tokenizer
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"✅ LoRA adapter saved to: {ADAPTER_DIR}")
print("   Files saved:")

import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"   - {f:40s} ({size/1e6:.1f} MB)")

## Step 10 — Test the Fine-Tuned Model on Medical Queries

In [ ]:
from unsloth import FastLanguageModel

# Switch model to inference mode (2x faster with Unsloth)
FastLanguageModel.for_inference(model)

def ask_medical_llm(question: str, max_new_tokens: int = 256) -> str:
    """Ask the fine-tuned medical LLM a question and return its response."""
    prompt = ALPACA_TEMPLATE.format(
        question=question,
        answer="",
        eos=""
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens    = max_new_tokens,
        use_cache         = True,
        temperature       = 0.3,
        do_sample         = True,
        repetition_penalty= 1.1,
    )

    # Decode only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# ── Test with sample medical queries ──────────────────────────────────────
test_questions = [
    "What are the main symptoms of Type 2 Diabetes?",
    "What is the mechanism of action of Metformin?",
    "How does hypertension affect the kidneys?",
    "What is the difference between systolic and diastolic blood pressure?",
    "What are the early warning signs of a myocardial infarction?",
]

print("=" * 60)
print("  FINE-TUNED MEDICAL LLM — TEST RESPONSES")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n[Q{i}] {question}")
    print("-" * 50)
    response = ask_medical_llm(question)
    print(f"[A{i}] {response}")
    print()

## Step 11 — (Optional) Save Merged 16-bit Model to Google Drive

In [ ]:
# Mount Google Drive to save your model persistently
from google.colab import drive
drive.mount("/content/drive")

GDRIVE_SAVE_PATH = "/content/drive/MyDrive/medical_llm_adapter"

# Save LoRA adapter only (small, ~100 MB)
model.save_pretrained(GDRIVE_SAVE_PATH)
tokenizer.save_pretrained(GDRIVE_SAVE_PATH)
print(f"✅ Adapter saved to Google Drive: {GDRIVE_SAVE_PATH}")

# OPTIONAL: Merge LoRA into full model and save in 16-bit (large, ~16 GB)
# model.save_pretrained_merged(
#     GDRIVE_SAVE_PATH + "_merged",
#     tokenizer,
#     save_method="merged_16bit"
# )
# print("Merged 16-bit model also saved.")

## Step 12 — (Optional) Push Adapter to HuggingFace Hub
Share your fine-tuned model with the community!

In [ ]:
# from huggingface_hub import login
# login(token="hf_your_token_here")  # Get from huggingface.co/settings/tokens

# REPO_NAME = "your-username/medical-llama3-qlora"
# model.push_to_hub(REPO_NAME, token="hf_your_token_here")
# tokenizer.push_to_hub(REPO_NAME, token="hf_your_token_here")
# print(f"✅ Model pushed to: https://huggingface.co/{REPO_NAME}")

print("Uncomment the lines above and add your HuggingFace token to push the model.")

---
## Summary

| Step | What was done |
|------|---------------|
| 1 | Installed Unsloth, TRL, PEFT, bitsandbytes |
| 2 | Verified T4 GPU availability |
| 3 | Loaded Llama-3.1-8B in **4-bit** with Unsloth |
| 4 | Attached **LoRA adapter** (r=16, target all projection layers) |
| 5 | Loaded **medical Q&A dataset**, formatted as Alpaca prompts |
| 6 | Configured **SFTTrainer** with 8-bit AdamW optimiser |
| 7 | Monitored **GPU VRAM** before & after |
| 8 | Ran **epoch-based training** |
| 9 | Saved **LoRA adapter** locally |
| 10 | **Tested** model on 5 new medical queries |
| 11 | Optionally saved to **Google Drive** |
| 12 | Optionally pushed to **HuggingFace Hub** |

### Key concepts practised
- **QLoRA** — quantised low-rank adaptation (4-bit base + 16-bit adapters)
- **PEFT** — parameter-efficient fine-tuning (only ~1% of params trained)
- **Unsloth** — 2x speed, 60% less VRAM versus vanilla HuggingFace training
- **SFTTrainer (TRL)** — supervised fine-tuning with instruction datasets